In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import cv2
import tensorflow as tf

from utils import generate_transition_linear

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

MODEL_PATH   = "/content/drive/MyDrive/HCI Project/models/bio_exergame_multimodal.keras"

SEGMENT_DIR  = "processed_data"
RESULT_DIR   = "result"
os.makedirs(RESULT_DIR, exist_ok=True)

TARGET_DURATION = 960

In [ ]:
# weights for different cost components
W_LEN      = 0.8
W_SMOOTH   = 0.89
W_VARIETY  = 0.001
W_INT      = 0.8
W_BIO      = 1.0

# simulated annealing params
MAX_ITER        = 1000
INIT_TEMP       = 1.0
FINAL_TEMP      = 0.01
COOLING_RATE    = 0.99

# model input config
T      = 16
H, W   = 128, 128
HR_LEN = 4
HR_CLASS_NAMES   = ["low", "medium", "high"]
DIFF_CLASS_NAMES = ["low", "medium", "high"]

In [ ]:
TARGET_HR_CLASS   = 1  # medium
TARGET_DIFF_CLASS = 1  # medium

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)

In [ ]:
def predict_segment_state(video_clip, hr_seq):
    """
    video_clip: (16,128,128,3) float32 in [0,1]
    hr_seq:     (4,) float32
    returns dict with hr_probs, diff_probs, perf
    """
    x = {
        "video_in": video_clip[None, ...],
        "hr_in":    hr_seq[None, ...],
    }
    hr_logits, diff_logits, perf = model.predict(x, verbose=0)

    return {
        "hr_probs":   hr_logits[0],
        "diff_probs": diff_logits[0],
        "perf":       float(perf[0, 0]),
    }

In [ ]:
def load_segments(segment_dir):
    files = [os.path.join(segment_dir, f)
             for f in sorted(os.listdir(segment_dir))
             if f.endswith(".npy")]
    segments = [np.load(f, allow_pickle=True).item() for f in files]
    for i, seg in enumerate(segments):
        seg["seg_idx"] = i
    return segments

def sample_clip_from_video(video_path, start_frame, end_frame, n_frames=T):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        # fallback: black clip
        cap.release()
        return np.zeros((n_frames, H, W, 3), dtype=np.float32)

    # sample indices evenly
    idxs = np.linspace(start_frame, max(start_frame, end_frame - 1),
                       n_frames, dtype=int)
    frames = []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok:
            frame = np.zeros((H, W, 3), dtype=np.uint8)
        else:
            frame = cv2.resize(frame, (W, H))
            frame = frame[..., ::-1]  # BGR -> RGB
        frames.append(frame.astype(np.float32) / 255.0)
    cap.release()
    return np.stack(frames, axis=0)

def synthetic_hr_seq_from_class(hr_class, hr_len=HR_LEN):
    # rough mapping for optimization (tune these values)
    SYNTH_HR_VALUES = {0: 80.0, 1: 115.0, 2: 150.0}
    base = SYNTH_HR_VALUES[int(hr_class)]
    return np.full((hr_len,), base, dtype=np.float32)

In [ ]:
def initial_pose(available_segments):
    idx = np.random.randint(len(available_segments))
    return available_segments[idx]

def targetlen_cost(sequence, duration, w_len):
    current_len = sum(len(seg['pred_position']) for seg in sequence)
    return w_len * (1.0 / duration) * abs(current_len - duration)

def normalize_displacement(d, scale=0.1):
    return (np.tanh(d * scale) + 1.0) / 2.0

def smoothness_cost(sequence):
    T_trans = len(sequence) - 1
    if T_trans <= 0:
        return float('inf')
    total_disp = 0.0
    for i in range(T_trans):
        last_pose  = sequence[i]['pred_position'][-1]
        first_pose = sequence[i+1]['pred_position'][0]
        disp = np.abs(last_pose - first_pose)
        total_disp += np.sum(disp)
    avg_disp = total_disp / T_trans
    return normalize_displacement(avg_disp)

def segment_id(seg):
    return np.mean(seg['pred_position'], axis=0)

def variety_cost(candidate_id, existing_ids):
    if candidate_id is None or not existing_ids:
        return 0.0
    dists = [np.linalg.norm(candidate_id - eid) for eid in existing_ids]
    return max(min(dists), 1e-2)

def intensity_cost(sequence):
    if len(sequence) == 0:
        return float('inf')
    total = 0.0
    M = np.ones(24)
    for seg in sequence:
        v   = np.diff(seg['pred_position'], axis=0)
        v2  = np.sum(v**2, axis=-1)
        ke  = 0.5 * np.sum(M * v2, axis=-1)
        total += np.sum(ke)
    return total

def segment_bio_cost(sequence):
    """
    For each segment, run the model on its (video, synthetic HR) window
    and penalize deviation from target classes + low predicted perf.
    (For speed, you may cache per-segment predictions.)
    """
    if len(sequence) == 0:
        return 0.0

    hr_devs, diff_devs, perf_vals = [], [], []

    for seg in sequence:
        video_path  = seg.get("video_path", None)
        start_frame = int(seg.get("start_frame", 0))
        end_frame   = int(seg.get("end_frame", start_frame + 120))  # ~4s @30fps
        hr_class    = int(seg.get("hr_class", TARGET_HR_CLASS))

        if video_path is None:
            continue

        video_clip = sample_clip_from_video(video_path, start_frame, end_frame)
        hr_seq     = synthetic_hr_seq_from_class(hr_class)

        pred = predict_segment_state(video_clip, hr_seq)

        hr_probs   = pred["hr_probs"]
        diff_probs = pred["diff_probs"]
        perf       = pred["perf"]

        hr_cls   = int(np.argmax(hr_probs))
        diff_cls = int(np.argmax(diff_probs))

        hr_devs.append(abs(hr_cls   - TARGET_HR_CLASS))
        diff_devs.append(abs(diff_cls - TARGET_DIFF_CLASS))
        perf_vals.append(perf)

    if len(hr_devs) == 0:
        return 0.0

    hr_cost   = float(np.mean(hr_devs))
    diff_cost = float(np.mean(diff_devs))
    perf_mean = float(np.mean(perf_vals))
    perf_cost = max(0.0, 1.0 - perf_mean)  # penalize perf < 1.0

    return hr_cost + diff_cost + perf_cost

def total_cost(sequence,
               duration,
               w_len,
               w_smooth,
               w_var,
               w_int,
               w_bio,
               candidate_id=None):
    existing_ids = [segment_id(seg) for seg in sequence]
    c_len   = targetlen_cost(sequence, duration, w_len)
    c_smooth= smoothness_cost(sequence) * w_smooth
    c_var   = variety_cost(candidate_id, existing_ids) * w_var
    c_int   = intensity_cost(sequence) * (1.0 - w_int)
    c_bio   = segment_bio_cost(sequence) * w_bio
    return c_len + c_smooth + c_var + c_int + c_bio

In [ ]:
def optimization_step(sequence,
                      available_segments,
                      duration,
                      w_len,
                      w_smooth,
                      w_var,
                      w_int,
                      w_bio,
                      temperature):
    current_cost = total_cost(sequence, duration, w_len, w_smooth, w_var,
                              w_int, w_bio, candidate_id=None)
    current_len  = sum(len(seg['pred_position']) for seg in sequence)
    trial_seq    = sequence.copy()
    candidate_id = None

    if len(trial_seq) == 0:
        move_type      = "add"
        selected_index = 0
    else:
        selected_index = random.randrange(len(trial_seq))
        if current_len >= duration:
            move_types = ["remove", "modify"]
            move_probs = [0.3, 0.7]
        else:
            move_types = ["add", "remove", "modify"]
            move_probs = [0.4, 0.2, 0.4]
        move_type = np.random.choice(move_types, p=move_probs)

    if move_type == "add" and current_len < duration:
        seg_add = random.choice(available_segments)
        before_or_after = random.choice(["before", "after"])
        if before_or_after == "before":
            trial_seq.insert(selected_index, seg_add)
        else:
            trial_seq.insert(selected_index + 1, seg_add)
        candidate_id = segment_id(seg_add)

    elif move_type == "remove" and len(trial_seq) > 1:
        trial_seq.pop(selected_index)

    elif move_type == "modify":
        seg_new = random.choice(available_segments)
        trial_seq[selected_index] = seg_new
        candidate_id = segment_id(seg_new)

    trial_cost = total_cost(trial_seq, duration, w_len, w_smooth, w_var,
                            w_int, w_bio, candidate_id=candidate_id)

    delta = trial_cost - current_cost
    accept_prob = np.exp(-delta / temperature)

    if (trial_cost < current_cost) or (np.random.random() < accept_prob):
        return trial_seq, trial_cost
    else:
        return sequence, current_cost

In [ ]:
def stitch_sequence(sequence, save_path):
    origin = np.zeros((1, 3))
    previous_end_root = origin

    final_pred = []
    final_root = []
    final_global = []

    for i, seg in enumerate(sequence):
        if i == 0:
            offset = origin - seg['root_position'][0]
        else:
            offset = previous_end_root - seg['root_position'][0]

        adj_root = seg['root_position'] + offset

        final_pred.extend(seg['pred_position'])
        final_root.extend(adj_root)

        previous_end_root = adj_root[-1]

        if i < len(sequence) - 1:
            next_seg = sequence[i+1]
            trans_pred = generate_transition_linear(
                seg['pred_position'][-1],
                next_seg['pred_position'][0],
                5
            )
            trans_root = generate_transition_linear(
                adj_root[-1],
                next_seg['root_position'][0] +
                (previous_end_root - next_seg['root_position'][0]),
                5
            )
            final_pred.extend(trans_pred)
            final_root.extend(trans_root)

    final_pred  = np.array(final_pred)
    final_root  = np.array(final_root)

    for pred, root in zip(final_pred, final_root):
        final_global.append(pred + root)
    final_global = np.array(final_global)

    data = {
        "pred_position": final_global,
        "root_position": final_root
    }
    np.save(save_path, data)

In [ ]:
if __name__ == "__main__":
    available_segments = load_segments(SEGMENT_DIR)

    current_sequence = [initial_pose(available_segments)]
    current_cost = total_cost(current_sequence,
                              TARGET_DURATION,
                              W_LEN,
                              W_SMOOTH,
                              W_VARIETY,
                              W_INT,
                              W_BIO,
                              candidate_id=None)

    temp      = INIT_TEMP
    iteration = 0

    while temp > FINAL_TEMP and iteration < MAX_ITER:
        print(f"Iter {iteration}, temp {temp:.4f}, "
              f"len {len(current_sequence)}, cost {current_cost:.4f}")
        current_sequence, current_cost = optimization_step(
            current_sequence,
            available_segments,
            TARGET_DURATION,
            W_LEN,
            W_SMOOTH,
            W_VARIETY,
            W_INT,
            W_BIO,
            temp
        )
        temp *= COOLING_RATE
        iteration += 1

    out_path = os.path.join(RESULT_DIR, "stitched_motion.npy")
    stitch_sequence(current_sequence, out_path)
    print("Saved stitched motion to:", out_path)